# 03 - Forecasting de Declinacion Productiva

Pipeline de entrenamiento para predecir declinacion de produccion por pozo usando **XGBoost** y features temporales de `src.features.temporal_features`.

Objetivos:
- Entrenar un regresor robusto para produccion mensual.
- Validar estabilidad temporal con `TimeSeriesSplit` (5 folds).
- Interpretar variables clave (`edad_pozo_meses`, lags).
- Definir criterio de cierre economico por margen operativo.


In [1]:
# Bootstrap de dependencias para este notebook
import importlib

required = ['scikit-learn', 'xgboost']
missing = []
for pkg, mod in [('scikit-learn', 'sklearn'), ('xgboost', 'xgboost')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    get_ipython().run_line_magic('pip', f"install {' '.join(missing)} -q")
    print('Instalado:', ', '.join(missing))
else:
    print('Dependencias OK')


Dependencias OK


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.features.temporal_features import (
    build_engine_from_env,
    load_monthly_well_production,
    build_feature_dataset,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# Configuracion de corrida (ajustable)
ANALYSIS_START_DATE = '2020-01-01'
ANALYSIS_END_DATE = '2027-01-01'
HISTORY_MONTHS = 6
TOP_POZOS = 1000  # subir gradualmente a 1500/2000 si el kernel responde estable


In [3]:
from xgboost import XGBRegressor


In [ ]:
engine = build_engine_from_env()
base_df = load_monthly_well_production(
    engine=engine,
    start_date=ANALYSIS_START_DATE,
    end_date=ANALYSIS_END_DATE,
    target_col='prod_pet',
    history_months=HISTORY_MONTHS,
    keep_only_window=False,
)

# Limite de pozos para estabilidad del kernel
top_pozos = (
    base_df.groupby('id_pozo', as_index=False)['target']
    .sum()
    .sort_values('target', ascending=False)
    .head(TOP_POZOS)['id_pozo']
)
base_df = base_df[base_df['id_pozo'].isin(top_pozos)].copy()

feature_df = build_feature_dataset(base_df, value_col='target', downcast=True)
feature_df = feature_df[feature_df['in_window']].copy()
feature_df = feature_df.sort_values(['fecha', 'id_pozo']).reset_index(drop=True)

print(f'Feature rows: {len(feature_df):,} | Pozos: {feature_df.id_pozo.nunique():,}')
feature_df.head()


In [ ]:
# Seleccion de features para modelado
base_features = [
    'profundidad',
    'edad_pozo_meses',
    'target_lag_1',
    'target_lag_3',
    'target_lag_6',
    'target_roll_mean_6',
    'target_roll_std_6',
]
reservorio_ohe = [c for c in feature_df.columns if c.startswith('res_')]
feature_cols = base_features + reservorio_ohe

model_df = feature_df.dropna(subset=['target_lag_6', 'target_roll_mean_6']).copy()
model_df = model_df.sort_values(['fecha', 'id_pozo']).reset_index(drop=True)

X = model_df[feature_cols].fillna(0)
y = model_df['target']

print(f'Registros modelables: {len(model_df):,}')
print(f'Cantidad de features: {len(feature_cols)}')


In [ ]:
# Validacion temporal (5 folds)
tscv = TimeSeriesSplit(n_splits=5)

metrics_rows = []
oof_pred = np.full(len(model_df), np.nan)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    oof_pred[test_idx] = pred

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    metrics_rows.append({
        'fold': fold,
        'train_min_fecha': model_df.iloc[train_idx]['fecha'].min(),
        'train_max_fecha': model_df.iloc[train_idx]['fecha'].max(),
        'test_min_fecha': model_df.iloc[test_idx]['fecha'].min(),
        'test_max_fecha': model_df.iloc[test_idx]['fecha'].max(),
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
    })

metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df)
print('Promedio CV -> RMSE: {:.4f} | MAE: {:.4f} | R2: {:.4f}'.format(
    metrics_df['rmse'].mean(),
    metrics_df['mae'].mean(),
    metrics_df['r2'].mean(),
))


In [ ]:
# Entrenamiento final con todo el dataset
final_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X, y)

model_df['pred_target'] = final_model.predict(X)


## Importancia de Variables
Confirmamos el peso relativo de `edad_pozo_meses` y los lags en la prediccion de declinacion.


In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': final_model.feature_importances_,
}).sort_values('importance', ascending=False)

top_n = 20
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(top_n), x='importance', y='feature', palette='viridis')
plt.title('Feature Importance - XGBoost (Top 20)')
plt.xlabel('Importancia relativa')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

display(importance_df[importance_df['feature'].isin(['edad_pozo_meses', 'target_lag_1', 'target_lag_3', 'target_lag_6'])])


## Limite Economico: Candidato a Cierre
Regla de negocio basada en margen mensual: `margen = prod_predicha * precio_bbl - opex_fijo`.


In [ ]:
def calcular_limite_economico(id_pozo, prod_predicha, precio_bbl=75, opex_fijo=15000):
    ingreso_estimado = prod_predicha * precio_bbl
    margen = ingreso_estimado - opex_fijo
    estado = 'Candidato a Cierre' if margen < 0 else 'Continuar Operacion'
    return {
        'id_pozo': id_pozo,
        'prod_predicha': prod_predicha,
        'precio_bbl': precio_bbl,
        'opex_fijo': opex_fijo,
        'margen_estimado': margen,
        'estado_pozo': estado,
    }

# Evaluacion en la ultima fecha disponible por pozo
latest_pred = (
    model_df.sort_values(['id_pozo', 'fecha'])
    .groupby('id_pozo', as_index=False)
    .tail(1)
)

economico_df = pd.DataFrame([
    calcular_limite_economico(row.id_pozo, row.pred_target)
    for row in latest_pred.itertuples(index=False)
])

display(economico_df['estado_pozo'].value_counts())
display(economico_df.sort_values('margen_estimado').head(15))


## Declinacion Real vs Predicha (Top 10 pozos historicos)
Se comparan curvas de los ultimos 12 meses para pozos de mayor produccion acumulada.


In [ ]:
top10_pozos = (
    model_df.groupby('id_pozo', as_index=False)['target']
    .sum()
    .sort_values('target', ascending=False)
    .head(10)['id_pozo']
)

plot_df = model_df[model_df['id_pozo'].isin(top10_pozos)].copy()
plot_df = plot_df.sort_values(['id_pozo', 'fecha'])
plot_df = plot_df.groupby('id_pozo', as_index=False).tail(12)

plot_long = plot_df.melt(
    id_vars=['id_pozo', 'fecha'],
    value_vars=['target', 'pred_target'],
    var_name='serie',
    value_name='produccion'
)
plot_long['serie'] = plot_long['serie'].map({'target': 'Real', 'pred_target': 'Predicha'})

g = sns.FacetGrid(
    plot_long,
    col='id_pozo',
    col_wrap=5,
    height=3.0,
    sharex=False,
    sharey=False,
)
g.map_dataframe(sns.lineplot, x='fecha', y='produccion', hue='serie', marker='o')
g.add_legend(title='Serie')
g.set_titles('Pozo {col_name}')
g.set_axis_labels('Fecha', 'Produccion')
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=45)
plt.subplots_adjust(top=0.92)
g.fig.suptitle('Curvas de Declinacion Real vs Predicha (Top 10 Pozos, ultimos 12 meses)')
plt.show()


In [ ]:
# Guardado del modelo
output_dir = Path('models')
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / 'xgboost_declinacion_v1.json'
final_model.save_model(model_path.as_posix())
print(f'Modelo guardado en: {model_path}')
